### Install Requirements

In [ ]:
%%bash
# show which pip/environment is being used
pip --version
python -c "import torch; print(\"Torch:\", torch.__version__, torch.__path__)"

The following will modify the standard azureml environment. But it seems to be the easiest way to install additional packages and keep bash magic working. If there is a problem, the package installation and command calling needs to be revised.

In [ ]:
!pip install -r requirements-no-version.txt

### Run Training - MVTec Test

In [ ]:
## datapath: mvtec ad dataset structure, each category has /train, /test, /fg_mask
## if /fg_mask does not exist, the code will create it and generate foreground masks for normal training images.
## augpath: augmented dtd dataset
## results_path: where the results are saved
## /mlruns: where mlflow tracking results are saved by default
## meta_epochs: 640 (set to be 5 here for faster training, 100 maybe enough)

In [ ]:
# adjust path!
%cd /home/azureuser/cloudfiles/code/Users/josbeth/GLASS


Test with MVTec data:

In [ ]:
%%bash

datapath=/home/azureuser/localfiles/data/mvtec_anomaly_detection
augpath=/home/azureuser/localfiles/data/dtd/images
classes=('bottle')
flags=($(for class in "${classes[@]}"; do echo '-d '"${class}"; done))

python main.py \
    --results_path /home/azureuser/localfiles/results/glass \
    --test ckpt \
  net \
    -b wideresnet50 \
    -le layer2 \
    -le layer3 \
    --meta_epochs 5 \
    --eval_epochs 1 \
    --limit 392 \
  dataset \
    --batch_size 16 \
    "${flags[@]}" mvtec $datapath $augpath

Sanity check for a foreground mask (should be a white circle which marks the foreground):

In [ ]:
from IPython.display import Image, display
# sanity check of fg_mask_image
display(Image('/home/azureuser/localfiles/data/mvtec_anomaly_detection/bottle/train/good/001.png', width=312))
display(Image('/home/azureuser/localfiles/data/mvtec_anomaly_detection/bottle/fg_mask/001.png', width=312))


### run test

In [ ]:
## use --test test to test the trained model

In [ ]:
%%bash

datapath=/home/azureuser/localfiles/data/mvtec_anomaly_detection
augpath=/home/azureuser/localfiles/data/dtd/images
classes=('bottle')
flags=($(for class in "${classes[@]}"; do echo '-d '"${class}"; done))

python main.py \
    --results_path /home/azureuser/localfiles/results/glass \
    --test test \
  net \
    -b wideresnet50 \
    -le layer2 \
    -le layer3 \
    --meta_epochs 5 \
    --eval_epochs 1 \
    --limit 392 \
  dataset \
    --batch_size 16 \
    "${flags[@]}" mvtec $datapath $augpath

### observe results

In [ ]:
## /results/models: saved model weights
## /results/eval: each image contains three parts from left to right: input image, groundtruth, predicted heatmap
## results.csv: evaluation results on all metrics for each category and the mean of them

In [ ]:
%cd /home/azureuser/localfiles/results/glass
!ls

In [ ]:
from IPython.display import Image, display
display(Image('eval/mvtec_bottle/001.png'))